# Robot session

Everything needed to bring the robot up, calibrate it, and hand coordinates to a
picking routine. Run the cells in order the first time; afterwards the
calibration sections can be re-run on their own.

The order in section 1 is not optional. `move_to_coordinates`, `move_relative`
and `get_position` all require both a run and a loaded pipette, so anything that
moves the robot fails until `create_run` and `load_pipette` have happened.

## 0. Setup

In [ ]:
import os
# Notebooks live one level below the repository root.
os.environ["MICROPICK_ROOT"] = os.path.abspath("..")

import time
import numpy as np
import cv2

from opentrons_api import ot2_api

from micropick import paths
from micropick.config import store
from micropick.config.schema import CameraSpec, PipetteOffset
from micropick.hardware import labware
from micropick.hardware.camera import CameraManager
from micropick.hardware.protocols import xyz
from micropick.core.calibration.pixel_map import PixelMap, compare_degrees
from micropick.workflows.calibrate_camera import calibrate_camera
from micropick.workflows.calibrate_pipette import TipDetector, calibrate_pipette_offset
from micropick.workflows.jog import JogController, Limits, jog_in_window

paths.ensure_layout()
print(paths.describe())
print("\nprofiles:", store.list_profiles())

### Profile

One profile per installation. Created once, then loaded on every later run.

In [ ]:
PROFILE = "lab_main"

profile = store.create_profile(PROFILE, camera_label="overview_cam",
                               notes="OT-2, gantry camera", exist_ok=True)
print(profile)
print("positions:", sorted(profile.positions) or "none yet")

### Cameras, first run only

Names must match what the operating system reports. Focus and exposure are
re-applied every time a camera is opened, which is what keeps a pixel map valid
across restarts: the map is only correct for the focus it was fitted at.

In [ ]:
from micropick.hardware import devices
print(*devices.list_devices(), sep="\n")

In [ ]:
profile.cameras = {
    "overview_cam": CameraSpec(
        device_name="20MP U3 Camera",
        resolutions=[[2592, 1944], [1920, 1080], [1280, 720]],
        default_resolution=[2592, 1944], fps=60, fourcc="MJPG",
        controls={"auto_exposure": "manual"},
        notes="on the gantry, manual focus ring"),
    "underview_cam": CameraSpec(
        device_name="Arducam B0478 (USB3 48MP)",
        resolutions=[[2000, 1500], [4000, 3000]],
        default_resolution=[4000, 3000], fps=30, fourcc="MJPG",
        controls={"autofocus": 0, "focus": 920, "auto_exposure": "manual"},
        notes="tip calibration module, motorised focus"),
}
profile.save_cameras()
print(*profile.cameras.values(), sep="\n")

## 1. Robot

In [ ]:
openapi = ot2_api.OpentronsAPI()
openapi.add_slot_offsets([5, 8, 9], (0, 0, 64.2))

In [ ]:
# Once after power on.
openapi.home_robot()

In [ ]:
openapi.create_run()
openapi.load_pipette()
print("run:", openapi.run_id, " pipette:", openapi.pipette_id)

### Labware

Definitions live in `labware/` and are uploaded into the current run. This has
to happen again after every `create_run`. Names and namespaces come from the
files themselves.

In [ ]:
for d in labware.list_definitions().values():
    print(d)

In [ ]:
labware.ensure_definitions(openapi)

TIP_RACK = "vwr_96_tiprack_200ul_xl"
labware.load_labware(openapi, TIP_RACK, 10)

In [ ]:
openapi.pick_up_tip(openapi.labware_dct["10"], "A4")

## 2. Cameras

In [ ]:
cams = CameraManager.from_profile(profile)
over_cam = cams.open("overview_cam")
print(over_cam)

In [ ]:
# The lower camera is only needed for tip calibration; open it there.
# cams.close("underview_cam")

### Jogging

Arrows or WASD move x and y, `q` and `e` move z, `+` and `-` change the step,
space saves a position, `u` undoes the last step, Enter finishes. The window
must have focus, so a stray keystroke in the notebook cannot drive the robot.

Limits are soft. Outside them the robot can always move back toward the working
area, only further out is refused.

In [ ]:
LIMITS = Limits(x=(0, 380), y=(0, 350), z=(0.1, 150))

def jog(title="", camera=None, step=1.0):
    ctrl = JogController(openapi, limits=LIMITS, step=step)
    pos = jog_in_window(ctrl, camera or over_cam, title=title)
    print("stopped at", tuple(round(v, 2) for v in pos))
    return pos

## 3. Camera calibration

Fits lens distortion and the camera-to-robot relationship together from one
sweep of a static ArUco marker. No chessboard and no undistortion stage.

Redo it after any change to focus, zoom, camera height, or the height of the
plane the objects sit on.

In [ ]:
MARKER_SIDE_MM = 6.8

aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_6X6_250)
params = cv2.aruco.DetectorParameters()
params.cornerRefinementMethod = cv2.aruco.CORNER_REFINE_SUBPIX
detector = cv2.aruco.ArucoDetector(aruco_dict, params)

Put the marker roughly in the centre of the frame and set Z to the height you
actually image the dish at. The sweep keeps whatever Z it starts from.

In [ ]:
jog("centre the marker, set the working Z, then Enter")

In [ ]:
pmap, report, sweep = calibrate_camera(
    openapi, over_cam, detector,
    marker_side_mm=MARKER_SIDE_MM, grid_n=7, degree=3,
    on_progress=lambda i, n: print(f"  {i}/{n}", end="\r"))

print("\n")
print(report)

`track side` should come out near the printed marker size. The fit never uses
it, so agreement is independent evidence the sweep was good.

Degrees 1 and 2 should give identical numbers: radial distortion is cubic in
image coordinates, so a quadratic reduces exactly to an affine fit. A difference
between them would mean something other than lens distortion is in the data.

In [ ]:
print(compare_degrees(sweep.track_px, sweep.gantry, sweep.image_size,
                      marker_side_mm=MARKER_SIDE_MM))

In [ ]:
profile.calibration.pixel_map = pmap.to_config()
profile.save_calibration()
sweep.save(str(paths.fixtures_dir() / f"sweep_{time.strftime('%Y%m%d_%H%M')}.npz"))
print("saved")

### Closed-loop check

Ask the map where the marker is, drive there, and see how far it lands from the
reference pixel. This is the only test that includes the robot.

Read the spread, not the absolute value. A consistent offset in one direction
with a small spread is the camera-to-tip constant and belongs to the pipette
offset; scatter is the map and the robot's repeatability.

In [ ]:
def marker_centre_now():
    frame = over_cam.read_after(time.monotonic())
    corners, ids, _ = detector.detectMarkers(frame)
    if ids is None or len(corners) == 0:
        return None
    return corners[0].reshape(4, 2).mean(axis=0)

origin = xyz(openapi)
errors = []
for dx, dy in [(0, 0), (10, 7), (-12, -8), (18, -11), (-20, 12)]:
    openapi.move_to_coordinates((origin[0] + dx, origin[1] + dy, origin[2]),
                                min_z_height=1, verbose=False)
    time.sleep(0.4)
    g = xyz(openapi)[:2]                       # read next to the frame
    q = marker_centre_now()
    if q is None or not pmap.covers(*q):
        print(f"({dx:+3.0f},{dy:+3.0f}) not usable")
        continue

    target = pmap.to_robot(q[0], q[1], g)
    openapi.move_to_coordinates((target[0], target[1], origin[2]),
                                min_z_height=1, verbose=False)
    time.sleep(0.4)
    q2 = marker_centre_now()
    if q2 is None:
        continue
    err_px = q2 - np.array(pmap.config.ref)
    scale = float(np.mean(pmap.mm_per_px(*q2)))
    errors.append(err_px * scale)
    print(f"({dx:+3.0f},{dy:+3.0f})  residual {err_px[0]:+7.1f}, {err_px[1]:+7.1f} px"
          f"  = {np.linalg.norm(err_px) * scale * 1000:6.0f} um")

if errors:
    e = np.array(errors)
    print(f"\nbias   {e.mean(0)[0]*1000:+.0f}, {e.mean(0)[1]*1000:+.0f} um"
          f"   (constant, belongs to the pipette offset)")
    print(f"spread {np.linalg.norm(e - e.mean(0), axis=1).max()*1000:.0f} um max"
          f"   (this is the map plus robot repeatability)")

## 4. Pipette offset calibration

Redo this whenever a tip is picked up: every tip seats differently.

The upper camera locates the crosshair disc, the robot drives there using the
current offset, and the lower camera measures how far the tip actually is. The
gantry is parked a few millimetres to one side first, otherwise the tip covers
the crosshair and neither can be measured.

On a new installation the offset must be filled in roughly by hand first,
measured with a ruler. The routine drives to where it thinks the target is
before looking, so an offset that is wrong by tens of millimetres puts the tip
outside the lower camera's view and the run cannot recover.

In [ ]:
from ultralytics import YOLO

tip_model = YOLO(str(paths.ml_models_dir() / profile.calibration.tip_target.model_file))
tip_detector = TipDetector(tip_model,
                           imgsz=profile.calibration.tip_target.imgsz,
                           conf=profile.calibration.tip_target.conf)
under_cam = cams.open("underview_cam")
print(under_cam)

First run only: fill in a rough offset measured with a ruler, and teach the
position of the calibration module.

In [ ]:
if profile.calibration.pipette_offset is None:
    profile.calibration.pipette_offset = PipetteOffset(
        dx=16.0, dy=60.0, tip_type="vwr_200ul_xl", method="manual")
    profile.save_calibration(backup=False)
print(profile.calibration.pipette_offset)

In [ ]:
# Teach where the crosshair disc is, once. Skip if it is already stored.
if "tip_calib" not in profile.positions:
    jog("bring the crosshair disc under the camera, then Enter")
    profile.remember("tip_calib", xyz(openapi))
print("tip_calib:", profile.where("tip_calib"))

In [ ]:
target = profile.calibration.tip_target
openapi.move_to_coordinates(profile.where("tip_calib"),
                            min_z_height=target.module_height - 0.1, verbose=False)
time.sleep(0.5)

`manual_touch_up` runs after the automatic correction, with the lower camera
live. Nudge the tip onto the crosshair with a small step if the result is not
good enough, then press Enter. Whatever it moves is included, because the offset
is read from the final pose rather than from the commanded moves.

In [ ]:
def touch_up(robot, camera, view):
    ctrl = JogController(robot, limits=LIMITS, step=0.05)
    jog_in_window(ctrl, camera, window="tip",
                  title="nudge the tip onto the crosshair, then Enter")

current = profile.calibration.pipette_offset
result = calibrate_pipette_offset(
    openapi, over_cam, under_cam, tip_detector, pmap,
    target=target,
    current_offset=(current.dx, current.dy),
    frames=7,
    tip_type=current.tip_type,
    manual_touch_up=touch_up)          # pass None to skip the manual step

print()
print(result)

In [ ]:
profile.calibration.pipette_offset = result.offset
profile.save_calibration(backup=False)
print("saved:", profile.calibration.pipette_offset)

## 5. Using the calibration

`pixel_to_robot` is the one function the picking code needs. The gantry pose has
to be read next to the frame the pixel came from: the pose is part of the
conversion, not a correction applied afterwards.

`mm_per_px` replaces the old global size ratio. Scale varies by several percent
across the frame, so a single number misreports objects near the edges.

In [ ]:
profile = store.load_profile(PROFILE)
profile.require_calibration()
pmap = PixelMap.from_config(profile.pixel_map)
off = profile.calibration.pipette_offset
tip_offset = np.array([off.dx, off.dy])

problems = profile.pixel_map.check_camera(over_cam.resolution)
if problems:
    raise RuntimeError("the calibration does not match the camera: " + "; ".join(problems))

def pixel_to_robot(u, v, gantry_xy):
    """Robot coordinates that put the pipette tip on the pixel (u, v)."""
    if not pmap.covers(u, v):
        raise ValueError(f"pixel ({u:.0f}, {v:.0f}) is outside the calibrated area")
    return pmap.to_robot(u, v, gantry_xy) + tip_offset

def area_mm2(area_px, u, v):
    su, sv = pmap.mm_per_px(u, v)
    return area_px * su * sv

print("ready:", pmap.config.degree, "degree map,",
      f"holdout {pmap.config.holdout_mean_um:.1f} um,",
      f"offset ({off.dx:.2f}, {off.dy:.2f}) mm")

In [ ]:
# Example: convert one detection.
g = xyz(openapi)[:2]
frame = over_cam.read_after(time.monotonic())
# u, v = ...detect something...
# tx, ty = pixel_to_robot(u, v, g)

## 6. Bridge to the old picking code

The picking state machine has not been ported yet. To run it against this
calibration, replace two things in the old notebook and leave the rest alone.

```python
# was: X, Y, _ = tf_mtx @ (cX, cY, 1), then a gantry delta added
g = xyz(openapi)[:2]              # read immediately before the frame
X, Y = pixel_to_robot(cX, cY, g)

# was: area_mm = area_px * size_conversion_ratio
area_mm = area_mm2(area_px, cX, cY)
```

Everything else, the routines, the logger, the floater check, stays as it is.

## 7. Shutting down

In [ ]:
openapi.retract_axis("leftZ")
cams.close_all()